# Point-prompted segmentation with INSID3-debiased DINOv3

This notebook shows how to prompt `insid3_point_prompt.py` with points. For each positive and negative click,
it takes the DINOv3 feature of the patch under that point and builds a similarity map:

$$S(p) = \overline{\cos(f_p, f_{\text{pos}})} - \overline{\cos(f_p, f_{\text{neg}})}$$

Here, $f$ are the **positionally debiased** features. The positional subspace is estimated by SVD from a
content-free image and projected out of the features.

**Plotting conventions**
- **Positives only:** only the similarity map is shown, with no threshold and no mask.
- **Positives + negatives:** the number of positives always equals the number of negatives, and the mask is $S > 0$.

**Sections**
1. Setup and feature extraction (encode once, reuse for every prompt)
2. One positive point
3. Several positive points
4. Adding negative points
5. Score distribution
6. Debiased vs. raw features: the positional bias
7. One-call API (`segment_from_points`)
8. *(Optional)* Interactive clicking

Needs `insid3_debias.py` and `insid3_point_prompt.py` on the Python path.

## 1. Setup

In [ ]:
import sys

sys.path.append(".")  # folder containing insid3_debias.py and insid3_point_prompt.py

# ── Config: edit these ──────────────────────────────────────────────
REPO_DIR       = "external/dinov3"                                  # local clone of facebookresearch/dinov3
WEIGHTS        = "pretrain_weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
MODEL_NAME     = "dinov3_vitl16"
IMAGE_PATH     = "assets/frame.jpg"
IMAGE_SIZE     = 768          # encoder input (int or (H, W)); ignored if PATCH_GRID is set
PATCH_GRID     = (67, 120)         # e.g. 48 or (40, 72) for 16:9 video
SVD_COMPONENTS = 500
THRESH         = 0.0          # used only when negatives are present (pos/neg decision boundary)
# ────────────────────────────────────────────────────────────────────

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from insid3_debias import INSID3Debias, load_dinov3_local
from insid3_point_prompt import segment_from_points, similarity_map

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
encoder = load_dinov3_local(REPO_DIR, MODEL_NAME, WEIGHTS, device=DEVICE)
model = INSID3Debias(encoder, image_size=IMAGE_SIZE, patch_grid=PATCH_GRID,
                     svd_components=SVD_COMPONENTS, use_bf16=(DEVICE == "cuda"), device=DEVICE)
print("encoder input:", model.image_size, "| patch grid:", model.patch_grid,
      "| positional basis:", tuple(model.positional_basis.shape))

Load the image and **encode it once**. Every prompt below reuses the cached features, so trying new
points costs only a matrix product.

In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
img = np.asarray(image)
H, W = img.shape[:2]

def frac(fx, fy):
    """Point given as a fraction of image width/height -> (x, y) in pixels."""
    return (fx * W, fy * H)

deb, raw = model(image, return_raw=True)   # (1, C, h, w) each, L2-normalized
feats_deb, feats_raw = deb[0], raw[0]
print(f"image {W}x{H}  ->  features {tuple(feats_deb.shape)}")

Plotting helper. It draws in-notebook instead of calling `save_vis`, because `save_vis` switches
matplotlib to the non-interactive `Agg` backend.

In [ ]:
def check_balanced(pos, neg):
    if neg and len(pos) != len(neg):
        raise ValueError(f"need equal positives and negatives, got {len(pos)} pos / {len(neg)} neg")

def run(pos, neg=(), feats=None, title=""):
    """Compute the similarity map from cached features and plot it.
    Positives only -> similarity map, no threshold.
    Positives + negatives (equal counts) -> similarity map + mask at S > THRESH (= 0)."""
    pos, neg = list(pos), list(neg)
    check_balanced(pos, neg)
    feats = feats_deb if feats is None else feats
    sim = similarity_map(feats, pos, neg, (H, W))
    mask = (sim > THRESH) if neg else None
    plot_result(sim, mask, pos, neg, title)
    return sim, mask

def draw_points(ax, pos, neg):
    if pos: ax.scatter(*zip(*pos), c="lime", s=110, marker="x", linewidths=3, label="positive")
    if neg: ax.scatter(*zip(*neg), c="red", s=110, marker="x", linewidths=3, label="negative")

def overlay(mask, color=(255, 140, 0), alpha=0.45):
    out = img.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array(color, np.float32)
    return out.astype(np.uint8)

def plot_result(sim, mask, pos, neg, title=""):
    n = 3 if neg else 2
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6 * H / W + 0.8))
    axes[0].imshow(img); axes[0].set_title("Prompts")
    axes[1].imshow(img)
    if neg:
        lim = np.abs(sim).max()
        im = axes[1].imshow(sim, cmap="viridis", alpha=0.3, vmin=-lim, vmax=lim)
        axes[1].contour(sim, levels=[THRESH], colors="k", linewidths=1)
        axes[1].set_title("Similarity (mean pos − mean neg)")
    else:
        im = axes[1].imshow(sim, cmap="viridis", alpha=0.3)
        axes[1].set_title("Similarity (mean pos)")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    if neg:
        axes[2].imshow(overlay(mask)); axes[2].contour(mask, levels=[0.5], colors="orange", linewidths=1)
        axes[2].set_title(f"Mask  S > {THRESH:g}   ({mask.mean() * 100:.1f}% of pixels)")
    for ax in axes:
        draw_points(ax, pos, neg); ax.axis("off")
    axes[0].legend(loc="lower right")
    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

Select whichever point you want from the image below to be used as a prompt.

In [ ]:
from utils import plot_frame

plot_frame(IMAGE_PATH)

## 2. One positive point

Put one point on the target, e.g. a tumor nodule. Without negatives, $S$ is the plain cosine similarity
in $[-1, 1]$. There is no natural decision boundary here (most tissue has *some* positive similarity to
any other tissue), so only the similarity map is shown, with no threshold.

In [ ]:
POS_1 = [(737, 134)]          # <- replace with a point on your target, e.g. [(512, 300)]

run(POS_1, title="One positive (no threshold)");

## 3. Several positive points

Averaging several positives, ideally spread over different parts of the target or over different
lesions, gives a less noisy query. A lesion you did not click on should also light up when it is
semantically similar. The debiasing is what makes that possible.

In [ ]:
POS_MULTI = [(115, 442), (794, 1032), (1301, 132)]   # <- edit

run(POS_MULTI, title=f"{len(POS_MULTI)} positives (no threshold)");

## 4. Adding negative points

Negatives are subtracted: $S = \overline{\text{sim}_{pos}} - \overline{\text{sim}_{neg}}$, with range $[-2, 2]$.
Now $S > 0$ means *closer to the positives than to the negatives*, so $\tau = 0$ is the natural
decision boundary and the mask is shown at $S > 0$. The number of negatives must equal the number of
positives. Put negatives on the distractors that leak into the mask: nearby healthy peritoneum,
instruments, specular highlights, fat.

In [ ]:
NEG = [(1594, 158), (1430, 653), (382, 547)]         # <- edit (same count as POS_MULTI)
check_balanced(POS_MULTI, NEG)

run(POS_MULTI, NEG, title=f"{len(POS_MULTI)} positives + {len(NEG)} negatives, τ = 0");

Negatives matter a lot. Adding prompt pairs one at a time (one positive + one negative each step, so the
counts stay equal) shows how each negative carves away a region:

In [ ]:
for k in range(1, len(NEG) + 1):
    run(POS_MULTI[:k], NEG[:k], title=f"{k} positive(s) + {k} negative(s), τ = 0")

## 5. Score distribution

The histogram of $S$ shows where the decision boundary $\tau = 0$ sits relative to the score distribution.
A clear bimodal split around 0 means the prompts separate target from background well.

In [ ]:
check_balanced(POS_MULTI, NEG)
sim = similarity_map(feats_deb, POS_MULTI, NEG, (H, W))
m = sim > THRESH

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sim.ravel(), bins=200, color="#2a78d6")
axes[0].axvline(THRESH, color="k", lw=1, ls="--", label=f"τ = {THRESH:g}")
axes[0].set_title("Distribution of S"); axes[0].set_xlabel("S"); axes[0].legend()
axes[1].imshow(overlay(m)); draw_points(axes[1], POS_MULTI, NEG)
axes[1].set_title(f"Mask  S > {THRESH:g}  ({m.mean() * 100:.1f}%)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Debiased vs. raw features: the positional bias

Raw DINOv3 patch features encode *where* a patch is as well as *what* it is. With one positive point,
the raw similarity map therefore shows a blob centered on the click that decays with distance, and a
same-looking lesion on the other side of the image scores low. After projecting out the positional
subspace, similarity should follow appearance instead of distance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6 * H / W + 0.6))
for ax, (name, f) in zip(axes, [("raw", feats_raw), ("debiased", feats_deb)]):
    s = similarity_map(f, POS_1, [], (H, W))
    ax.imshow(img); im = ax.imshow(s, cmap="viridis", alpha=0.6)
    fig.colorbar(im, ax=ax, fraction=0.046)
    draw_points(ax, POS_1, []); ax.set_title(f"{name} features — one positive"); ax.axis("off")
plt.tight_layout(); plt.show()

And the full pos/neg setup with raw features, for comparison with section 4:

In [ ]:
run(POS_MULTI, NEG, feats=feats_raw, title="RAW features, positives + negatives, τ = 0");